# Spectrogram Generation

In this notebook, the previously labeled vibration time-series segments are transformed into time–frequency representations using short-time Fourier transform (STFT). The resulting spectrograms are normalized and converted into RGB images by stacking the X, Y, and Z acceleration axes. These images serve as input for the subsequent convolutional neural network (CNN) used for chatter detection.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import spectrogram
from PIL import Image
import os
from tqdm import tqdm
import random
import h5py

# In-/ Output Structure & Class-Mapping

In [3]:
INPUT_ROOT = Path(r"F:\20_Datensätze\03_Airbus_Dataset\dftrain.h5")
OUTPUT_ROOT = Path("../../data/05_airbus_dataset")

# Spectrogram configuration
- NPERSEG: controls frequency resolution of spectrogram
larger values → better frequency resolution, lower time resolution
- MAX_FREQ_CUT: limits analysis to relevant chatter frequency range
- DB_MIN / DB_MAX: spectrogram is converted to decibel scale --> fixed scaling for energy
- TARGET_IMG_SIZE: image size of three color spectrogram

In [14]:
# segmentation of timeseries  
WINDOW_SEC = 5    

# config of spectrogram
NPERSEG = 512
FS = 1024                 # sampling
MAX_FREQ_CUT = 1024      # max. freq.
DB_MIN = -55             
DB_MAX = -6              
EPS = 1e-12

# output image size
TARGET_IMG_SIZE = (150, 100) 

WINDOW_SIZE = int(WINDOW_SEC * FS)
STRIDE = WINDOW_SIZE // 2

## Spectrogram generator function

In [5]:
# =========================================================
# LOAD DATA
# =========================================================
def load_data(file_path):
    df = pd.read_hdf(file_path, key='dftrain')
    return df

# =========================================================
# SPECTROGRAM
# =========================================================
def create_spectrogram(sig, fs):
    noverlap = int(0.75 * NPERSEG)

    f, t, Sxx = spectrogram(
        sig,
        fs=fs,
        nperseg=NPERSEG,
        noverlap=noverlap,
        mode="psd"
    )

    # Frequency limitation
    mask = f <= MAX_FREQ_CUT
    Sxx = Sxx[mask, :]

    # dB-Scaling
    Sxx_db = 10 * np.log10(Sxx + EPS)
    Sxx_db = np.clip(Sxx_db, DB_MIN, DB_MAX)

    # Normalization
    Sxx_norm = 1.0 - (Sxx_db - DB_MIN) / (DB_MAX - DB_MIN)

    return Sxx_norm

def process_file(file_path):
    data = load_data(file_path)

    x = data[:, 0] / 1000
    y = data[:, 1] / 1000
    z = data[:, 2] / 1000

    x_segments = segment_signal(x)
    y_segments = segment_signal(y)
    z_segments = segment_signal(z)

    relative_path = file_path.relative_to(INPUT_ROOT)

    for i, (xs, ys, zs) in enumerate(zip(x_segments, y_segments, z_segments)):

        specs = [
            create_spectrogram(xs, FS),
            create_spectrogram(ys, FS),
            create_spectrogram(zs, FS)
        ]

        rgb = np.stack(specs, axis=-1)
        rgb = np.clip(rgb, 0, 1)
        rgb = np.flipud(rgb)

        img = Image.fromarray((rgb * 255).astype(np.uint8))
        img = img.resize(TARGET_IMG_SIZE)

        # Ordner basierend auf Dateinamen erstellen
        file_output_dir = OUTPUT_ROOT / relative_path.parent / file_path.stem

        # kompletter Pfad inkl. Dateiname
        out_path = file_output_dir / f"{file_path.stem}_seg{i}.png"

        # Ordner sicher erstellen
        out_path.parent.mkdir(parents=True, exist_ok=True)
        img.save(out_path)


def segment_signal(sig):
    segments = []
    
    for start in range(0, len(sig) - WINDOW_SIZE, WINDOW_SIZE):
        segment = sig[start:start + WINDOW_SIZE]
        segments.append(segment)
    
    return segments


In [ ]:
def create_spectrogram(sig, fs):
    noverlap = int(0.75 * NPERSEG)

    f, t, Sxx = spectrogram(
        sig,
        fs=fs,
        nperseg=NPERSEG,
        noverlap=noverlap,
        mode="psd"
    )

    mask = f <= MAX_FREQ_CUT
    Sxx = Sxx[mask, :]

    Sxx_db = 10 * np.log10(Sxx + EPS)
    Sxx_db = np.clip(Sxx_db, DB_MIN, DB_MAX)

    return 1.0 - (Sxx_db - DB_MIN) / (DB_MAX - DB_MIN)


# RGB Image Conversion
This function converts the labeled vibration time-series segment into a 3-channel spectrogram image.
Combines three spectrograms into a single 3-channel image:

R = X-axis
G = Y-axis
B = Z-axis

Each vibration segment is transformed into a three-channel spectrogram image, where each channel corresponds to a different sensor axis. This encoding preserves spatial vibration directionality while enabling convolutional neural networks to learn cross-axis correlations relevant for chatter detection.

**Advantages of these representation:**
- harmonic stripes (chatter)
- broadband noise (instability)
- axis coupling effects

In [15]:

def process_dataframe(df):
    
    for i in tqdm(range(len(df))):
                
        sig = df.iloc[i].values.astype(float)

        for start in range(0, len(sig) - WINDOW_SIZE, STRIDE):
            segment = sig[start:start + WINDOW_SIZE]

            spec = create_spectrogram(segment, FS)

            spec = np.flipud(spec)

            img = Image.fromarray((spec * 255).astype(np.uint8))
            img = img.resize(TARGET_IMG_SIZE)

            out_path = OUTPUT_ROOT / f"airbus_{i}_seg{start}.png"
            out_path.parent.mkdir(parents=True, exist_ok=True)

            img.save(out_path)



In [ ]:

file = r"F:\20_Datensätze\03_Airbus_Dataset\dftrain.h5"

dftrain = pd.read_hdf(file, key='dftrain')

print("Shape:", dftrain.shape)

process_dataframe(dftrain[0:3])


Shape: (1677, 61440)


100%|██████████| 1677/1677 [01:38<00:00, 17.01it/s]


In [11]:
dftrain#.head()

,0,1,2,3,4,5,6,7,8,9,...,61430,61431,61432,61433,61434,61435,61436,61437,61438,61439
0,0.041259,0.041259,0.032573,0.023887,0.029315,0.041259,0.045602,0.038001,0.030401,0.032573,...,0.034744,-0.007600,-0.003257,0.065145,0.047773,-0.009772,0.031487,0.096632,0.077089,0.040173
1,-0.211722,-0.264924,-0.274696,-0.236694,-0.156349,-0.059716,0.005429,0.046687,0.153091,0.281210,...,0.260581,-0.004343,-0.241037,-0.636252,-0.953292,-0.980436,-0.846888,-0.838202,-0.880546,-0.739398
2,0.214105,0.154930,0.136640,0.013987,-0.038733,-0.015063,-0.111894,-0.104363,0.047340,-0.054871,...,0.040884,0.375490,0.699337,0.965085,1.086662,1.132926,1.279249,1.296464,0.937112,0.451879
3,-0.154837,-0.127768,-0.217638,-0.284770,-0.299929,-0.270694,-0.077960,0.092036,0.076877,0.154837,...,-0.286936,-0.171079,-0.036814,-0.024904,0.031400,0.140761,-0.011911,-0.173244,-0.063884,0.081208
4,-1.022780,-0.916376,-0.676425,-0.461445,-0.330069,-0.122690,0.178064,0.489675,0.799115,0.931577,...,1.009751,1.134613,0.836030,0.479903,0.109661,-0.285553,-0.628651,-0.916376,-1.010837,-0.804544
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1672,-0.063989,0.017228,0.015997,-0.321177,-0.503300,-0.535295,-0.562367,-0.600515,-0.573442,-0.317485,...,-0.121826,-0.134131,-0.111981,-0.057836,-0.073834,-0.125517,-0.082448,-0.036917,-0.114442,-0.183354
1673,0.993037,0.726052,0.341885,0.286314,0.535177,0.605245,0.198124,-0.375711,-0.579875,-0.495310,...,0.038658,0.188460,0.273025,0.045907,-0.108727,0.072484,0.415577,0.356382,-0.096646,-0.280273
1674,0.570550,0.253578,-0.193841,-0.502279,-0.657108,-0.838757,-0.971642,-0.881427,-0.700996,-0.520566,...,-0.146295,0.128008,0.162143,0.018287,-0.017068,0.087777,0.062175,-0.041450,-0.140199,-0.247482
1675,0.630677,0.605010,0.477897,0.155225,-0.441229,-0.679566,-0.515786,-0.424118,-0.183336,0.394784,...,0.229781,-0.041556,-0.176003,-0.301894,-0.177225,0.033001,0.025667,0.088001,0.206559,0.003667
